# ⚡ Centro de Control de Empleo — Carlos Molina Gil
**Perfil:** Técnico de Oficina Técnica Eléctrica (EPLAN P8 / AutoCAD) | Automatización & Robótica | Diseño Mecánico (Solid Edge / Siemens NX)

Este **Jupyter Notebook** se sincroniza automáticamente con tu carpeta de **Google Drive** y extrae **ofertas de trabajo reales en tiempo real en Valencia y alrededores**.

### ¿Qué hace este Notebook?
0. **Sincronización Automática con Google Drive:** Comprueba si has editado tu CV en Drive (`1XIZPcGc0eDDvf0mdzb8RgVVgOQ5Dt4Oh`), descarga los PDFs/imágenes nuevos si han cambiado y extrae su texto automáticamente.
1. **Lee tu CV** (`../cv/mi_cv.md` y el texto directo de `CV_Carlos_Molina_Gil_1Pagina.pdf`).
2. **Buscador Masivo de Ofertas Reales en Valencia:** Extrae en tiempo real decenas de ofertas activas en **Valencia** (Paterna, Riba-roja, Albal, Silla, Llíria, Sagunto, Almussafes, etc.) para *Oficina Técnica Eléctrica, EPLAN, AutoCAD, Automatización Industrial, PLC, Solid Edge y Cuadros Eléctricos*, las puntúa según tu CV y las guarda en `../ofertas/ofertas_guardadas.json`.
3. **Analizador de Compatibilidad (Match %) y Generador de Prompts:** Elige cualquiera de las ofertas descargadas (o pega una nueva) y genera al instante el prompt perfecto para tu carta de presentación y entrevista.

## Paso 0: ☁️ Sincronización Automática con tu Carpeta de Google Drive
Al ejecutar esta celda, el Notebook se conecta a tu Google Drive y comprueba si has subido una versión nueva de tu CV.

In [1]:
import importlib
import sincronizar_drive
importlib.reload(sincronizar_drive)

# Verifica tus archivos sincronizados de Drive y extrae automáticamente el texto del último PDF
archivos_actualizados = sincronizar_drive.sincronizar_cv_drive()

✅ Conectado a Google Drive: tu carpeta local ya tiene la última versión de todos tus archivos.
📂 Carpeta local del CV: C:\Users\marcm\.gemini\antigravity\scratch\Repo-de-prueba\busqueda_empleo\cv
   📄 CV_Carlos_Molina_Gil_1Pagina.pdf (308.0 KB) — Última sincronización: 2026-11-21 15:18:21
   📄 CV_Carlos_Molina_Gil_2Paginas.pdf (314.4 KB) — Última sincronización: 2026-11-21 15:18:23
   🖼️ Imágenes y fotos disponibles: 5 archivos (Foto_Carlos_Camiseta_Retocada.jpg, Foto_Carlos_Polo_Azul.jpg, Imagen_CV_1Pagina.png, Imagen_CV_2Paginas_Pag1.png, Imagen_CV_2Paginas_Pag2.png)


✅ Texto extraído del PDF actualizado correctamente (5238 caracteres).


## Paso 1: 📄 Cargar tu CV y Perfil Técnico actualizado
Lee tanto `mi_cv.md` como el texto extraído directamente de tu PDF más reciente (`CV_Carlos_Molina_Gil_1Pagina.pdf`).

In [2]:
import json
import urllib.parse
from pathlib import Path
from datetime import datetime
from pypdf import PdfReader
import requests
from bs4 import BeautifulSoup

RUTA_CV_MD = Path('../cv/mi_cv.md')
RUTA_CV_PDF = Path('../cv/CV_Carlos_Molina_Gil_1Pagina.pdf')
RUTA_OFERTAS = Path('../ofertas/ofertas_guardadas.json')

texto_cv = ''
if RUTA_CV_PDF.exists():
    reader = PdfReader(str(RUTA_CV_PDF))
    texto_pdf = '\n'.join(p.extract_text() or '' for p in reader.pages)
    print(f'✅ PDF leído directamente ({len(texto_pdf)} caracteres): {RUTA_CV_PDF.name}')
    texto_cv = texto_pdf

if RUTA_CV_MD.exists():
    texto_md = RUTA_CV_MD.read_text(encoding='utf-8')
    print(f'✅ Markdown de CV cargado ({len(texto_md)} caracteres): {RUTA_CV_MD.name}\n')
    print('--- VISTA PREVIA DE TU PERFIL Y FORMACIÓN ---')
    for linea in texto_md.splitlines():
        if any(palabra in linea for palabra in ['CFGS', 'Grado', 'Bachillerato', 'Dositec', 'DMVAL']):
            print(' •', linea.strip('- '))

✅ PDF leído directamente (5238 caracteres): CV_Carlos_Molina_Gil_1Pagina.pdf
✅ Markdown de CV cargado (6624 caracteres): mi_cv.md

--- VISTA PREVIA DE TU PERFIL Y FORMACIÓN ---
 • **Dositec (Grupo Binoovo)** — *Rafelbunyol (Valencia)* | **Oct. 2024 – Nov. 2026**
 • **DMVAL Assemblies Electrical S.L.** — *Manises (Valencia)* | **Jun. 2024 – Sept. 2024**
 • **CFGS en Programación de la Producción en Fabricación Mecánica** (2022 – 2024)
 • **Grado en Ingeniería Telemática (1.er curso superado)** (2021 – 2022)
 • **CFGS en Automatización y Robótica Industrial** (2018 – 2021)
 • **Bachillerato Científico-Tecnológico** (2016 – 2018)


## Paso 2: 🎯 Configurar tus Búsquedas y Palabras Clave en Valencia
Aquí están configuradas **8 líneas de búsqueda específicas para tu perfil** en la provincia de Valencia, junto con las palabras clave para puntuar y ordenar las mejores ofertas para ti.

In [3]:
UBICACION = 'Valencia, Comunidad Valenciana, España'
UBICACION_CORTA = 'Valencia'

# 8 búsquedas específicas basadas en tu experiencia (Dositec, DMVAL, MetalWorld, Ford) y tus 2 CFGS
BUSQUEDAS_VALENCIA = [
    'Oficina Tecnica Electrica',
    'EPLAN Electrico',
    'Proyectista Electrico',
    'Delineante Electrico AutoCAD',
    'Automatizacion Industrial PLC',
    'Oficina Tecnica Maquinaria',
    'Diseño Mecanico Solid Edge NX',
    'Tecnico Electrico Industrial Cuadros'
]

# Palabras clave para puntuar el título y categoría de cada oferta encontrada
KEYWORDS_ALTO_VALOR = [
    'oficina técnica', 'oficina tecnica', 'eplan', 'eléctrico', 'electrico', 'eléctrica', 'electrica',
    'proyectista', 'delineante', 'autocad', 'automatización', 'automatizacion', 'automation',
    'plc', 'robótica', 'robotica', 'robotics', 'maquinaria', 'solid edge', 'nx', 'cad',
    'mecánico', 'mecanico', 'industrial', 'mantenimiento', 'cuadros', 'instalaciones', 'ingeniero', 'técnico', 'tecnico'
]

SKILLS_CV = [
    'eplan', 'autocad', 'solid edge', 'siemens nx', 'neumática', 'neumatica',
    'bom', 'atex', 'hmi', 'tia portal', 'plc', 'allen-bradley', 'omron',
    'esquemas eléctricos', 'maquinaria', 'cuadros eléctricos', 'fluidsim'
]

print(f'📍 Ubicación objetivo: {UBICACION}')
print(f'🔎 Líneas de búsqueda activas ({len(BUSQUEDAS_VALENCIA)}): {BUSQUEDAS_VALENCIA}')

📍 Ubicación objetivo: Valencia, Comunidad Valenciana, España
🔎 Líneas de búsqueda activas (8): ['Oficina Tecnica Electrica', 'EPLAN Electrico', 'Proyectista Electrico', 'Delineante Electrico AutoCAD', 'Automatizacion Industrial PLC', 'Oficina Tecnica Maquinaria', 'Diseño Mecanico Solid Edge NX', 'Tecnico Electrico Industrial Cuadros']


## Paso 3: 🔄 Recargar Ofertas de Trabajo Reales en Valencia (En Tiempo Real)
Cada vez que ejecutes esta celda, el Notebook consultará en directo las ofertas activas en **Valencia y área metropolitana** para tus 8 categorías, eliminará duplicados, calculará tu puntuación de afinidad (`Score`) y guardará todas las ofertas en `../ofertas/ofertas_guardadas.json`.

In [4]:
def recargar_ofertas_valencia(busquedas, ubicacion=UBICACION, max_por_busqueda=12):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
        'Accept-Language': 'es-ES,es;q=0.9'
    }
    ofertas_unicas = {}
    print(f'🔄 Buscando ofertas reales en vivo en {UBICACION_CORTA} para {len(busquedas)} especialidades...\n')

    for query in busquedas:
        url = (
            f'https://www.linkedin.com/jobs-guest/jobs/api/seeMoreJobPostings/search'
            f'?keywords={urllib.parse.quote(query)}&location={urllib.parse.quote(ubicacion)}&start=0'
        )
        try:
            resp = requests.get(url, headers=headers, timeout=10)
            if resp.status_code != 200:
                continue
            soup = BeautifulSoup(resp.text, 'html.parser')
            cards = soup.find_all('div', class_='base-card')[:max_por_busqueda]
            print(f'   ✅ "{query}": {len(cards)} ofertas encontradas en Valencia')

            for card in cards:
                t_el = card.find('h3', class_='base-search-card__title')
                c_el = card.find('h4', class_='base-search-card__subtitle')
                l_el = card.find('span', class_='job-search-card__location')
                d_el = card.find('time')
                a_el = card.find('a', class_='base-card__full-link')

                titulo = t_el.get_text(strip=True) if t_el else 'Sin título'
                empresa = c_el.get_text(strip=True) if c_el else 'Empresa en Valencia'
                lugar = l_el.get_text(strip=True) if l_el else UBICACION_CORTA
                fecha = d_el.get('datetime', 'Reciente') if d_el else 'Reciente'
                enlace = a_el.get('href', '').split('?')[0] if a_el else ''

                clave = f'{titulo.lower()}_{empresa.lower()}'
                texto_eval = f'{titulo} {query}'.lower()
                coincidencias = [k for k in KEYWORDS_ALTO_VALOR if k in texto_eval]

                # Bonus de puntuación para tus áreas estrella
                score = len(set(coincidencias)) * 12
                if any(w in titulo.lower() for w in ['oficina técnica', 'oficina tecnica', 'eplan', 'eléctric', 'electric', 'proyectista', 'delineante', 'automatización', 'automatizacion', 'plc', 'mecánic', 'mecanic']):
                    score += 35
                score = min(99, max(45, score))

                if clave not in ofertas_unicas:
                    ofertas_unicas[clave] = {
                        'titulo': titulo,
                        'empresa': empresa,
                        'ubicacion': lugar,
                        'fecha_publicacion': fecha,
                        'categoria_busqueda': query,
                        'afinidad_estimada': f'{score}%',
                        'score_num': score,
                        'url': enlace
                    }
        except Exception as e:
            print(f'   ⚠️ Error en búsqueda "{query}": {e}')

    lista_ordenada = sorted(ofertas_unicas.values(), key=lambda x: (x['score_num'], x['fecha_publicacion']), reverse=True)
    return lista_ordenada

def generar_radar_portales(busquedas, ubicacion='Valencia'):
    radar = []
    for puesto in busquedas:
        q_puesto = urllib.parse.quote(puesto)
        q_loc = urllib.parse.quote(ubicacion)
        q_combo = urllib.parse.quote(f'{puesto} {ubicacion}')
        radar.append({
            'puesto': puesto,
            'linkedin_filtro_semana': f'https://www.linkedin.com/jobs/search/?keywords={q_puesto}&location={q_loc}&f_TPR=r604800',
            'infojobs_valencia': f'https://www.infojobs.net/jobsearch/search-results/list.xhtml?keyword={q_puesto}&provinceIds=46',
            'indeed_valencia': f'https://es.indeed.com/jobs?q={q_puesto}&l={q_loc}&fromage=7',
            'google_jobs': f'https://www.google.com/search?q={q_combo}+empleo&ibp=htl;jobs'
        })
    return radar

# 1. Ejecutar descarga masiva de ofertas reales en Valencia
ofertas_valencia = recargar_ofertas_valencia(BUSQUEDAS_VALENCIA)
radar_portales = generar_radar_portales(BUSQUEDAS_VALENCIA, UBICACION_CORTA)

# 2. Guardar todas las ofertas en ofertas_guardadas.json
datos_guardados = {
    'ultima_recarga': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'candidato': 'Carlos Molina Gil',
    'ubicacion': UBICACION,
    'total_ofertas_unicas_encontradas': len(ofertas_valencia),
    'ofertas_reales_valencia': ofertas_valencia,
    'enlaces_directos_portales': radar_portales
}
RUTA_OFERTAS.write_text(json.dumps(datos_guardados, indent=2, ensure_ascii=False), encoding='utf-8')

print(f'\n🎉 ¡ÉXITO! Se han encontrado y guardado {len(ofertas_valencia)} OFERTAS REALES ÚNICAS en Valencia y alrededores.')
print(f'💾 Archivo actualizado: {RUTA_OFERTAS.resolve()}\n')
print('=== 🏆 TOP 25 OFERTAS REALES EN VALENCIA PARA TU PERFIL ===')
for idx, of in enumerate(ofertas_valencia[:25], 1):
    print(f"{idx:2d}. [{of['afinidad_estimada']}] {of['titulo']} — {of['empresa']}")
    print(f"    📍 {of['ubicacion']} | 📅 {of['fecha_publicacion']} | 🏷️ {of['categoria_busqueda']}")
    print(f"    🔗 {of['url']}\n")

🔄 Buscando ofertas reales en vivo en Valencia para 8 especialidades...



   ✅ "Oficina Tecnica Electrica": 10 ofertas encontradas en Valencia


   ✅ "EPLAN Electrico": 10 ofertas encontradas en Valencia


   ✅ "Proyectista Electrico": 10 ofertas encontradas en Valencia


   ✅ "Delineante Electrico AutoCAD": 10 ofertas encontradas en Valencia


   ✅ "Automatizacion Industrial PLC": 10 ofertas encontradas en Valencia


   ✅ "Oficina Tecnica Maquinaria": 10 ofertas encontradas en Valencia


   ✅ "Tecnico Electrico Industrial Cuadros": 10 ofertas encontradas en Valencia

🎉 ¡ÉXITO! Se han encontrado y guardado 54 OFERTAS REALES ÚNICAS en Valencia y alrededores.
💾 Archivo actualizado: C:\Users\marcm\.gemini\antigravity\scratch\Repo-de-prueba\busqueda_empleo\ofertas\ofertas_guardadas.json

=== 🏆 TOP 25 OFERTAS REALES EN VALENCIA PARA TU PERFIL ===
 1. [99%] Técnico/a electromecánico — MOBA SPAIN - Residuos
    📍 Valencia/València | 📅 2026-11-17 | 🏷️ Tecnico Electrico Industrial Cuadros
    🔗 https://es.linkedin.com/jobs/view/t%C3%A9cnico-a-electromec%C3%A1nico-at-moba-spain-4485822450

 2. [99%] Técnico/a Electromecánico — Grupo La Plana
    📍 Onda | 📅 2026-10-30 | 🏷️ Tecnico Electrico Industrial Cuadros
    🔗 https://es.linkedin.com/jobs/view/t%C3%A9cnico-a-electromec%C3%A1nico-at-grupo-la-plana-4474691642

 3. [99%] Ingeniero Eléctrico o Técnico de Instalaciones Eléctricas — Electricidad y Comunicaciones Quart SLU
    📍 Aldaia | 📅 2026-10-26 | 🏷️ Oficina Tecnica Electrica
 

## Paso 4: 🎯 Seleccionar una Oferta Real y Generar tu Prompt de Candidatura
Cambia el número `INDICE_OFERTA_ELEGIDA` (por ejemplo `1` para la primera oferta de la lista de arriba, `2` para la segunda, etc.) para extraer su descripción completa y generar tu Prompt personalizado.

In [5]:
# 👇 Elige el número de oferta de la lista de arriba (1, 2, 3...)
INDICE_OFERTA_ELEGIDA = 1

if ofertas_valencia and 1 <= INDICE_OFERTA_ELEGIDA <= len(ofertas_valencia):
    seleccion = ofertas_valencia[INDICE_OFERTA_ELEGIDA - 1]
    print(f"🎯 Has seleccionado la oferta #{INDICE_OFERTA_ELEGIDA}: {seleccion['titulo']} en {seleccion['empresa']}")
    print(f"🔗 Enlace: {seleccion['url']}\n")

    prompt = f"""=== 📋 COPIA ESTE PROMPT Y ENVÍAMELO EN EL CHAT DE ANTIGRAVITY ===
Analiza mi CV en `busqueda_empleo/cv/mi_cv.md` para postular a esta oferta real que he encontrado en el Notebook:
- Puesto: {seleccion['titulo']}
- Empresa: {seleccion['empresa']}
- Ubicación: {seleccion['ubicacion']}
- Categoría: {seleccion['categoria_busqueda']}
- Enlace de la oferta: {seleccion['url']}

Por favor:
1. Dime qué puntos concretos de mi experiencia en Dositec (EPLAN P8, AutoCAD, neumática, BOM, ATEX, HMI), DMVAL, MetalWorld (Siemens NX) o Ford España debo destacar.
2. Redacta un mensaje corto y directo para contactar por LinkedIn con RRHH o el responsable técnico de {seleccion['empresa']}.
3. Prepara 3 posibles preguntas técnicas de entrevista para este puesto en {seleccion['empresa']}.
=================================================================="""
    print(prompt)
else:
    print('⚠️ Elige un número válido de la lista de ofertas.')

🎯 Has seleccionado la oferta #1: Técnico/a electromecánico en MOBA SPAIN - Residuos
🔗 Enlace: https://es.linkedin.com/jobs/view/t%C3%A9cnico-a-electromec%C3%A1nico-at-moba-spain-4485822450

=== 📋 COPIA ESTE PROMPT Y ENVÍAMELO EN EL CHAT DE ANTIGRAVITY ===
Analiza mi CV en `busqueda_empleo/cv/mi_cv.md` para postular a esta oferta real que he encontrado en el Notebook:
- Puesto: Técnico/a electromecánico
- Empresa: MOBA SPAIN - Residuos
- Ubicación: Valencia/València
- Categoría: Tecnico Electrico Industrial Cuadros
- Enlace de la oferta: https://es.linkedin.com/jobs/view/t%C3%A9cnico-a-electromec%C3%A1nico-at-moba-spain-4485822450

Por favor:
1. Dime qué puntos concretos de mi experiencia en Dositec (EPLAN P8, AutoCAD, neumática, BOM, ATEX, HMI), DMVAL, MetalWorld (Siemens NX) o Ford España debo destacar.
2. Redacta un mensaje corto y directo para contactar por LinkedIn con RRHH o el responsable técnico de MOBA SPAIN - Residuos.
3. Prepara 3 posibles preguntas técnicas de entrevista par